In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from metpy.io import Level2File

# Radar

In [ ]:
filename = "KATX20231017_232552_V06"

radar = Level2File(filename)

print(radar)

In [ ]:
print(radar.sweeps)

In [ ]:
f = radar

In [ ]:
import numpy as np

sweep = f.sweeps[0]

azimuth = np.array([r.header.az_angle for r in sweep])
elevation = np.array([r.header.el_angle for r in sweep])

reflectivity = np.array([
    r.moments[b"REF"][1]
    for r in sweep
])

print("Shape:", reflectivity.shape)
print("Azimuth:", azimuth.shape)
print("Elevation:", elevation.shape)

In [ ]:
import matplotlib.pyplot as plt

range_km = (
    sweep[0].moments[b"REF"][0].first_gate
    + np.arange(reflectivity.shape[1])
    * sweep[0].moments[b"REF"][0].gate_width
)

plt.figure(figsize=(10, 7))

plt.pcolormesh(
    azimuth,
    range_km,
    reflectivity.T,
    shading="auto",
)

plt.xlabel("Azimuth (degrees)")
plt.ylabel("Range (km)")
plt.title("KATX Reflectivity — 0.5° Elevation Sweep")
plt.colorbar(label="Reflectivity (dBZ)")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Geod

# Radar location from the Level-II file
radar_lat = 48.1946106
radar_lon = -122.4956970

# Maximum/unambiguous range from the radar metadata
range_km = 467

# Create a geodesic circle around the radar
geod = Geod(ellps="WGS84")

azimuths = np.linspace(0, 360, 361)

lons, lats, _ = geod.fwd(
    np.full_like(azimuths, radar_lon),
    np.full_like(azimuths, radar_lat),
    azimuths,
    np.full_like(azimuths, range_km * 1000)
)

# Make the map
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Map extent around the radar
ax.set_extent([-130, -115, 40, 55], crs=ccrs.PlateCarree())

# Background
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linewidth=0.5)

# Radar coverage
ax.plot(
    lons,
    lats,
    transform=ccrs.PlateCarree(),
    linewidth=1.5,
    label="467 km radar range"
)

# Radar location
ax.plot(
    radar_lon,
    radar_lat,
    marker="^",
    markersize=8,
    transform=ccrs.PlateCarree(),
    label="KATX"
)

ax.legend()
ax.set_title("KATX NEXRAD Radar Coverage")

plt.show()

In [ ]:
sweep = f.sweeps[0]

azimuths = np.array([
    radial.header.az_angle
    for radial in sweep
])

print(azimuths.min())
print(azimuths.max())
print(len(azimuths))

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.set_extent([-130, -115, 40, 55])

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linewidth=0.5)

for az in azimuths:
    end_lon, end_lat, _ = geod.fwd(
        radar_lon,
        radar_lat,
        az,
        range_km * 1000
    )

    ax.plot(
        [radar_lon, end_lon],
        [radar_lat, end_lat],
        transform=ccrs.PlateCarree(),
        linewidth=0.2
    )

ax.plot(
    radar_lon,
    radar_lat,
    "^",
    markersize=8,
    transform=ccrs.PlateCarree()
)

ax.set_title("KATX NEXRAD 0.5° Sweep")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from metpy.calc import azimuth_range_to_lat_lon
from metpy.units import units

# First elevation sweep
sweep = 0

# Azimuth angle of each radial
az = np.array([
    ray[0].az_angle
    for ray in f.sweeps[sweep]
]) * units.degrees

# Reflectivity metadata
ref_hdr = f.sweeps[sweep][0][4][b'REF'][0]

# Range of each radar gate
ref_range = (
    np.arange(ref_hdr.num_gates + 1) * ref_hdr.gate_width
    + ref_hdr.first_gate
) * units.kilometers

# Reflectivity values
ref = np.array([
    ray[4][b'REF'][1]
    for ray in f.sweeps[sweep]
])

# Mask missing data
ref = np.ma.masked_invalid(ref)

# Radar location
radar_lon = f.sweeps[0][0][1].lon
radar_lat = f.sweeps[0][0][1].lat

# Convert radar azimuth/range to geographic coordinates
lon, lat = azimuth_range_to_lat_lon(
    az,
    ref_range,
    radar_lon,
    radar_lat
)

# Map
fig = plt.figure(figsize=(12, 10))

ax = plt.axes(
    projection=ccrs.PlateCarree()
)

ax.set_extent(
    [-130, -115, 40, 55],
    crs=ccrs.PlateCarree()
)

# Background
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linewidth=0.5)

# Radar reflectivity
mesh = ax.pcolormesh(
    lon,
    lat,
    ref,
    transform=ccrs.PlateCarree(),
    cmap="turbo",
    vmin=-10,
    vmax=30
)

# Radar location
ax.plot(
    radar_lon,
    radar_lat,
    "^",
    markersize=8,
    color="black",
    transform=ccrs.PlateCarree()
)

# Colorbar
plt.colorbar(
    mesh,
    ax=ax,
    label="Reflectivity (dBZ)"
)

ax.set_title(
    f"KATX Radar Reflectivity — {f.dt}"
)

plt.show()

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
# First elevation sweep
sweep = 0

# Azimuth angle of each radial
az = np.array([
    ray[0].az_angle
    for ray in f.sweeps[sweep]
]) * units.degrees

# Reflectivity metadata
ref_hdr = f.sweeps[sweep][0][4][b'REF'][0]

# Range of each radar gate
ref_range = (
    np.arange(ref_hdr.num_gates + 1) * ref_hdr.gate_width
    + ref_hdr.first_gate
) * units.kilometers

# Reflectivity values
ref = np.array([
    ray[4][b'REF'][1]
    for ray in f.sweeps[sweep]
])

# Mask missing data
ref = np.ma.masked_invalid(ref)

# Radar location
radar_lon = f.sweeps[0][0][1].lon
radar_lat = f.sweeps[0][0][1].lat

# Convert radar azimuth/range to geographic coordinates
lon, lat = azimuth_range_to_lat_lon(
    az,
    ref_range,
    radar_lon,
    radar_lat
)

# Map
fig = plt.figure(figsize=(12, 10))

ax = plt.axes(
    projection=ccrs.PlateCarree()
)

ax.set_extent(
    [-130, -115, 40, 55],
    crs=ccrs.PlateCarree()
)

# Background
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linewidth=0.5)

colors = [
    "#646464",  # < -10
    "#04e9e7",  # -10 to 0
    "#019ff4",  # 0 to 5
    "#0300f4",  # 5 to 10
    "#02fd02",  # 10 to 15
    "#01c501",  # 15 to 20
    "#008e00",  # 20 to 25
    "#fdf802",  # 25 to 30
    "#e5bc00",  # 30 to 35
    "#fd9500",  # 35 to 40
    "#fd0000",  # 40 to 45
    "#d40000",  # 45 to 50
    "#bc0000",  # 50 to 55
    "#ff00ff",  # 55+
]

bounds = [-10, 0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 80]

cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, cmap.N)

mesh = ax.pcolormesh(
    lon,
    lat,
    ref,
    transform=ccrs.PlateCarree(),
    cmap=cmap,
    norm=norm
)

plt.colorbar(
    mesh,
    ax=ax,
    label="Reflectivity (dBZ)",
    ticks=bounds
)

ax.set_title(
    f"KATX Radar Reflectivity — {f.dt}"
)

plt.show()

# WRF

In [ ]:
ds = xr.open_dataset("WAC00WG-01/wrfout_d02_processed_23101700.nc")
ds

In [ ]:
precip_mm = ds["tp"] * 1000
precip_mm
plt.plot(precip_mm, ds["t"])

In [ ]:
ds["t"]
time = ds["t"]
time

In [ ]:
# Read data
data = pd.read_csv('2023-05-12.csv')

# Rename columns
data = data.rename(columns=str.lower)
print(data.columns)

# Keep only station_code == 129 (Pink Mountain) & 131 (Muskwa) & 120 (Silver)
data = data[data['station_code'].isin([129, 131, 120])]

# Convert date_time column to datetime type
data['date_time'] = pd.to_datetime(data['date_time'], format='%Y%m%d%H')

# Keep only date_time and temperature columns
data = data[['date_time', 'station_code', 'temperature']]

In [ ]:
data

In [ ]:
# Prepare data for verif
data_verif = data.copy()

data_verif['date'] = data_verif['date_time'].dt.strftime('%Y%m%d')
data_verif['hour'] = data_verif['date_time'].dt.hour
data_verif['leadtime'] = 0

# print(data_verif['date'].head())
# print(data_verif['hour'].head())
# print(data_verif['leadtime'].head())

In [ ]:
data_verif

In [ ]:
# Plotting
# Select variables to plot
# vars = [col for col in data.columns if col not in ['date_time', 'station_code']]
vars = ['temperature', 'relative_humidity', 'wind_speed', 'precipitation']
fig, axes = plt.subplots(nrows=len(vars), ncols=1, figsize=(12, 4 * len(vars)), sharex=True)
for ax, var in zip(axes, vars):
    ax.plot(data['date_time'], data[var], label=var)
    ax.set_ylabel(var)
    ax.legend()
    ax.grid(True)

axes[-1].set_xlabel('Time')
plt.suptitle('Variables over Time', fontsize=14)
plt.tight_layout()
plt.show()